# 🇱🇰 Sri Lanka Population Prediction — Fixed Pipeline## National & District Level Predictions (Overfitting-Free)This notebook:- Uses **real yearly data** (not fake interpolated monthly data)- Proper **Train / Validation / Test** splits- Shows all splits on **same graph** for comparison- Computes **RMSE, MAE, MAPE, MASE, R²** metrics- Predicts both **national** and **district** populations given a date

In [ ]:
# ============================================================# CELL 2: Setup & Install Dependencies# ============================================================!pip install -q pandas numpy scikit-learn statsmodels matplotlib seaborn openpyxl xlrd pmdarimaimport warningswarnings.filterwarnings('ignore')import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.linear_model import LinearRegression, Ridgefrom sklearn.svm import SVRfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scoreimport pickleimport os# For auto ARIMAfrom pmdarima import auto_arimafrom statsmodels.tsa.holtwinters import ExponentialSmoothingplt.style.use('seaborn-v0_8-darkgrid')sns.set_palette("husl")print("✅ All dependencies loaded!")

In [ ]:
# ============================================================# CELL 3: Upload / Mount Data# ============================================================# OPTION A: Upload files manually (default for Colab)# OPTION B: Mount Google DriveMODE = "upload"  # Change to "drive" if using Google DriveDRIVE_PATH = "/content/drive/MyDrive/Population/data/raw"if MODE == "upload":    from google.colab import files    print("📁 Upload ALL raw data files from data/raw/ folder:")    print("  1. Sri_Lanka_Population_1950_2025.xlsx")    print("  2. Sri_Lanka_Birth_Rate_1950_2025.xlsx")    print("  3. Year,Deaths per 1000 People.csv")    print("  4. Sri-Lanka-Population-Density-People-per-Square-KM-2026-03-05-21-40.csv")    print("  5. Sri_Lanka_Rural_Population_1960_2023.xlsx")    print("  6. Sri_Lanka_Urban_Population_1960_2023.xlsx")    print("  7. sri_lanka_district_population_2014_2024_new.csv")    uploaded = files.upload()    RAW_DIR = "."elif MODE == "drive":    from google.colab import drive    drive.mount('/content/drive')    RAW_DIR = DRIVE_PATHelse:    RAW_DIR = "data/raw"  # localprint(f"\n✅ Data directory: {RAW_DIR}")

In [ ]:
# ============================================================# CELL 4: Load & Merge All National Raw Data (YEARLY)# ============================================================pop_df = pd.read_excel(f"{RAW_DIR}/Sri_Lanka_Population_1950_2025.xlsx")birth_df = pd.read_excel(f"{RAW_DIR}/Sri_Lanka_Birth_Rate_1950_2025.xlsx")death_df = pd.read_csv(f"{RAW_DIR}/Year,Deaths per 1000 People.csv")density_df = pd.read_csv(f"{RAW_DIR}/Sri-Lanka-Population-Density-People-per-Square-KM-2026-03-05-21-40.csv")rural_df = pd.read_excel(f"{RAW_DIR}/Sri_Lanka_Rural_Population_1960_2023.xlsx")urban_df = pd.read_excel(f"{RAW_DIR}/Sri_Lanka_Urban_Population_1960_2023.xlsx")# Standardize column namespop_df.columns = ["Year", "National_Population"]birth_df.columns = ["Year", "Birth_Rate"]death_df.columns = ["Year", "Death_Rate"]density_df.columns = ["Year", "Population_Density"]rural_df.columns = ["Year", "Rural_Population"]urban_df.columns = ["Year", "Urban_Population"]# Merge all on Yearnat_df = pop_df.copy()for df in [birth_df, death_df, density_df, rural_df, urban_df]:    nat_df = nat_df.merge(df, on="Year", how="outer")nat_df = nat_df.sort_values("Year").reset_index(drop=True)nat_df = nat_df.interpolate(method='linear', limit_direction='both')nat_df = nat_df.dropna(subset=["National_Population"])print(f"✅ National yearly data: {len(nat_df)} rows, Years {nat_df['Year'].min()}-{nat_df['Year'].max()}")print(f"   Columns: {list(nat_df.columns)}")nat_df.head(10)

In [ ]:
# ============================================================# CELL 5: Data Exploration — Visualize All Features# ============================================================fig, axes = plt.subplots(3, 2, figsize=(16, 14))fig.suptitle("Sri Lanka — Raw Yearly Data Overview", fontsize=18, fontweight='bold')features = [    ("National_Population", "Population", "steelblue"),    ("Birth_Rate", "Birth Rate (per 1000)", "coral"),    ("Death_Rate", "Death Rate (per 1000)", "indianred"),    ("Population_Density", "Pop. Density (per km²)", "seagreen"),    ("Rural_Population", "Rural Population", "olive"),    ("Urban_Population", "Urban Population", "purple"),]for ax, (col, title, color) in zip(axes.flat, features):    valid = nat_df.dropna(subset=[col])    ax.plot(valid["Year"], valid[col], 'o-', color=color, markersize=3, linewidth=1.5)    ax.set_title(title, fontsize=13, fontweight='bold')    ax.set_xlabel("Year")    ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig("national_data_exploration.png", dpi=150, bbox_inches='tight')plt.show()print("✅ Data exploration plots saved")

## 🔴 Why Current Models Were Overfitting**The Problem**: The old pipeline converted 76 yearly points → 900 monthly points via **linear interpolation**.Every month's value was `year_value + (month/12) * (next_year - year_value)` — perfectly smooth, no real variation.Models could achieve 99.99% accuracy because they just learned to draw straight lines between known points.This is **memorization, not learning**.**The Fix**: Train on **real yearly data** (76 actual observations), then forecast at any granularity needed.

In [ ]:
# ============================================================# CELL 7: Show Why Interpolated Data Causes Overfitting# ============================================================# Demonstrate the problem: interpolate a small segment to monthlysample = nat_df[(nat_df["Year"] >= 2010) & (nat_df["Year"] <= 2020)].copy()monthly_fake = []for i in range(len(sample) - 1):    y1, p1 = sample.iloc[i]["Year"], sample.iloc[i]["National_Population"]    y2, p2 = sample.iloc[i+1]["Year"], sample.iloc[i+1]["National_Population"]    for m in range(1, 13):        frac = (m - 1) / 12.0        monthly_fake.append({"Year": y1 + frac, "Pop": p1 + frac * (p2 - p1)})mf = pd.DataFrame(monthly_fake)fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))fig.suptitle("Why Linear Interpolation Causes Fake Accuracy", fontsize=15, fontweight='bold')ax1.plot(sample["Year"], sample["National_Population"]/1e6, 'ro', markersize=10, label="Real Yearly Data", zorder=5)ax1.plot(mf["Year"], mf["Pop"]/1e6, 'b-', alpha=0.6, linewidth=1, label="Interpolated Monthly (FAKE)")ax1.set_title("Real vs Interpolated Data (2010-2020)")ax1.set_ylabel("Population (Millions)")ax1.legend()ax1.grid(True, alpha=0.3)# Show residuals are zeroax2.bar(range(len(mf)), [0]*len(mf), color='green', alpha=0.5)ax2.set_title("Residuals on Interpolated Data = ZERO\n(Model just connects dots → fake 99.9% accuracy)")ax2.set_ylabel("Prediction Error")ax2.set_xlabel("Monthly data points")plt.tight_layout()plt.savefig("overfitting_diagnosis.png", dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# ============================================================# CELL 8: Evaluation Metrics Helper Functions# ============================================================def calc_rmse(y_true, y_pred):    return float(np.sqrt(mean_squared_error(y_true, y_pred)))def calc_mae(y_true, y_pred):    return float(mean_absolute_error(y_true, y_pred))def calc_mape(y_true, y_pred):    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)    mask = y_true != 0    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)def calc_mase(y_true, y_pred, y_train, seasonal=1):    y_train = np.array(y_train, dtype=float)    if len(y_train) <= seasonal:        return np.nan    scale = np.mean(np.abs(y_train[seasonal:] - y_train[:-seasonal]))    if scale == 0:        return np.nan    return float(mean_absolute_error(y_true, y_pred) / scale)def calc_r2(y_true, y_pred):    return float(r2_score(y_true, y_pred))def eval_metrics(y_true, y_pred, y_train, seasonal=1):    return {        "RMSE": calc_rmse(y_true, y_pred),        "MAE": calc_mae(y_true, y_pred),        "MAPE": calc_mape(y_true, y_pred),        "MASE": calc_mase(y_true, y_pred, y_train, seasonal),        "R²": calc_r2(y_true, y_pred),    }print("✅ Evaluation functions defined")

In [ ]:
# ============================================================# CELL 9: Proper Train / Validation / Test Split (YEARLY)# ============================================================# National: 76 years → Train(1950-2009), Val(2010-2017), Test(2018-2025)train_nat = nat_df[nat_df["Year"] <= 2009].copy()val_nat = nat_df[(nat_df["Year"] >= 2010) & (nat_df["Year"] <= 2017)].copy()test_nat = nat_df[nat_df["Year"] >= 2018].copy()print(f"Train: {len(train_nat)} years ({train_nat['Year'].min()}-{train_nat['Year'].max()})")print(f"Val:   {len(val_nat)} years ({val_nat['Year'].min()}-{val_nat['Year'].max()})")print(f"Test:  {len(test_nat)} years ({test_nat['Year'].min()}-{test_nat['Year'].max()})")# Visualize the splitfig, ax = plt.subplots(figsize=(16, 5))ax.plot(train_nat["Year"], train_nat["National_Population"]/1e6, 'bo-', label=f"Train ({len(train_nat)} pts)", markersize=4)ax.plot(val_nat["Year"], val_nat["National_Population"]/1e6, 'o-', color='orange', label=f"Validation ({len(val_nat)} pts)", markersize=6)ax.plot(test_nat["Year"], test_nat["National_Population"]/1e6, 'ro-', label=f"Test ({len(test_nat)} pts)", markersize=6)ax.axvline(x=2009.5, color='orange', linestyle='--', alpha=0.7, label='Train/Val boundary')ax.axvline(x=2017.5, color='red', linestyle='--', alpha=0.7, label='Val/Test boundary')ax.set_xlabel("Year", fontsize=12)ax.set_ylabel("Population (Millions)", fontsize=12)ax.set_title("National Population — Train / Validation / Test Split", fontsize=14, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig("national_split.png", dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# ============================================================# CELL 10: Feature Engineering for ML Models# ============================================================feature_cols = ["Year", "Birth_Rate", "Death_Rate", "Population_Density", "Rural_Population", "Urban_Population"]feature_cols = [c for c in feature_cols if c in nat_df.columns]X_train = train_nat[feature_cols].valuesy_train = train_nat["National_Population"].valuesX_val = val_nat[feature_cols].valuesy_val = val_nat["National_Population"].valuesX_test = test_nat[feature_cols].valuesy_test = test_nat["National_Population"].values# Scale features for SVR and Ridgescaler = StandardScaler()X_train_sc = scaler.fit_transform(X_train)X_val_sc = scaler.transform(X_val)X_test_sc = scaler.transform(X_test)print(f"Features: {feature_cols}")print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

In [ ]:
# ============================================================# CELL 11: Train National Models# ============================================================models = {}predictions = {}# 1. Linear Regressionlr = LinearRegression()lr.fit(X_train, y_train)models["Linear"] = lrpredictions["Linear"] = {    "train": lr.predict(X_train),    "val": lr.predict(X_val),    "test": lr.predict(X_test),}# 2. Ridge Regression (regularized — prevents overfitting)ridge = Ridge(alpha=1000.0)ridge.fit(X_train_sc, y_train)models["Ridge"] = ridgepredictions["Ridge"] = {    "train": ridge.predict(X_train_sc),    "val": ridge.predict(X_val_sc),    "test": ridge.predict(X_test_sc),}# 3. SVR (Support Vector Regression — good for small datasets)svr = SVR(kernel='rbf', C=1e10, epsilon=50000, gamma='scale')svr.fit(X_train_sc, y_train)models["SVR"] = svrpredictions["SVR"] = {    "train": svr.predict(X_train_sc),    "val": svr.predict(X_val_sc),    "test": svr.predict(X_test_sc),}# 4. Auto ARIMA (time series — no feature leakage)ts_train = pd.Series(y_train, index=pd.date_range(start=f"{int(train_nat['Year'].min())}", periods=len(y_train), freq='YS'))arima_model = auto_arima(ts_train, seasonal=False, stepwise=True, suppress_warnings=True,                         max_p=5, max_q=5, max_d=2, information_criterion='aic')models["ARIMA"] = arima_modelarima_train_pred = arima_model.predict_in_sample()arima_val_pred = arima_model.predict(n_periods=len(y_val))# For test, refit on train+valts_trainval = pd.Series(np.concatenate([y_train, y_val]),    index=pd.date_range(start=f"{int(train_nat['Year'].min())}", periods=len(y_train)+len(y_val), freq='YS'))arima_refit = auto_arima(ts_trainval, seasonal=False, stepwise=True, suppress_warnings=True,                          max_p=5, max_q=5, max_d=2)arima_test_pred = arima_refit.predict(n_periods=len(y_test))predictions["ARIMA"] = {"train": arima_train_pred, "val": arima_val_pred, "test": arima_test_pred}# 5. Exponential Smoothing (Holt's linear trend)ets = ExponentialSmoothing(ts_train, trend='add', seasonal=None).fit(optimized=True)models["ETS"] = etsets_train_pred = ets.fittedvalues.valuesets_val_pred = ets.forecast(len(y_val)).valuesets_refit = ExponentialSmoothing(ts_trainval, trend='add', seasonal=None).fit(optimized=True)ets_test_pred = ets_refit.forecast(len(y_test)).valuespredictions["ETS"] = {"train": ets_train_pred, "val": ets_val_pred, "test": ets_test_pred}print("✅ All 5 national models trained!")for name in models:    print(f"  - {name}")

In [ ]:
# ============================================================# CELL 12: Combined Train/Val/Test Comparison Plot (ALL MODELS)# ============================================================model_names = list(predictions.keys())n_models = len(model_names)fig, axes = plt.subplots(2, 3, figsize=(22, 12))fig.suptitle("National Population — All Models: Train / Validation / Test", fontsize=18, fontweight='bold')years_train = train_nat["Year"].valuesyears_val = val_nat["Year"].valuesyears_test = test_nat["Year"].valuesfor idx, name in enumerate(model_names):    ax = axes.flat[idx]    preds = predictions[name]    # Actual data    ax.plot(years_train, y_train/1e6, 'b.', alpha=0.4, markersize=4, label="Actual (Train)")    ax.plot(years_val, y_val/1e6, '.', color='orange', markersize=6, label="Actual (Val)")    ax.plot(years_test, y_test/1e6, 'r.', markersize=6, label="Actual (Test)")    # Predictions    ax.plot(years_train, preds["train"]/1e6, 'b-', linewidth=1.5, alpha=0.7, label="Pred (Train)")    ax.plot(years_val, preds["val"]/1e6, '-', color='orange', linewidth=2, label="Pred (Val)")    ax.plot(years_test, preds["test"]/1e6, 'r-', linewidth=2, label="Pred (Test)")    # Split boundaries    ax.axvline(x=2009.5, color='gray', linestyle='--', alpha=0.5)    ax.axvline(x=2017.5, color='gray', linestyle='--', alpha=0.5)    # Metrics on plot    val_mape = calc_mape(y_val, preds["val"])    test_mape = calc_mape(y_test, preds["test"])    ax.set_title(f"{name}\nVal MAPE={val_mape:.2f}% | Test MAPE={test_mape:.2f}%", fontsize=12, fontweight='bold')    ax.set_xlabel("Year")    ax.set_ylabel("Population (M)")    ax.grid(True, alpha=0.3)# Last subplot: legendaxes.flat[-1].legend(*axes.flat[0].get_legend_handles_labels(), loc='center', fontsize=12)axes.flat[-1].set_title("Legend", fontsize=12)axes.flat[-1].axis('off')plt.tight_layout()plt.savefig("national_all_models_comparison.png", dpi=150, bbox_inches='tight')plt.show()print("✅ Combined comparison plot saved!")

In [ ]:
# ============================================================# CELL 13: Evaluation Metrics Table (All Models × All Splits)# ============================================================rows = []for name in predictions:    preds = predictions[name]    for split_name, y_true, y_pred in [("Train", y_train, preds["train"]),                                        ("Validation", y_val, preds["val"]),                                        ("Test", y_test, preds["test"])]:        m = eval_metrics(y_true, y_pred, y_train, seasonal=1)        m["Model"] = name        m["Split"] = split_name        rows.append(m)metrics_df = pd.DataFrame(rows)[["Model", "Split", "RMSE", "MAE", "MAPE", "MASE", "R²"]]print("\n📊 National Population — Model Evaluation Metrics")print("=" * 90)# Display as styled tablestyled = metrics_df.style.format({    "RMSE": "{:,.0f}", "MAE": "{:,.0f}", "MAPE": "{:.2f}%",    "MASE": "{:.3f}", "R²": "{:.4f}"}).background_gradient(subset=["MAPE"], cmap="RdYlGn_r")display(styled)# Savemetrics_df.to_csv("national_metrics.csv", index=False)# Overfitting checkprint("\n🔍 Overfitting Check (Val MAPE / Train MAPE ratio):")for name in predictions:    train_mape = calc_mape(y_train, predictions[name]["train"])    val_mape = calc_mape(y_val, predictions[name]["val"])    ratio = val_mape / max(train_mape, 0.01)    status = "✅ OK" if ratio < 5 else "⚠️ Check" if ratio < 10 else "🔴 Overfitting"    print(f"  {name:15s}: Train={train_mape:.2f}%, Val={val_mape:.2f}%, Ratio={ratio:.1f}x  {status}")

In [ ]:
# ============================================================# CELL 14: Residual Analysis# ============================================================fig, axes = plt.subplots(2, 3, figsize=(22, 10))fig.suptitle("National Population — Residual Analysis", fontsize=16, fontweight='bold')for idx, name in enumerate(model_names):    ax = axes.flat[idx]    preds = predictions[name]    # Residuals for each split    res_train = (y_train - preds["train"]) / 1e6    res_val = (y_val - preds["val"]) / 1e6    res_test = (y_test - preds["test"]) / 1e6    ax.bar(years_train, res_train, color='steelblue', alpha=0.6, width=0.8, label='Train')    ax.bar(years_val, res_val, color='orange', alpha=0.8, width=0.8, label='Val')    ax.bar(years_test, res_test, color='crimson', alpha=0.8, width=0.8, label='Test')    ax.axhline(y=0, color='black', linewidth=0.5)    ax.set_title(f"{name}", fontsize=11, fontweight='bold')    ax.set_ylabel("Residual (M)")    ax.grid(True, alpha=0.3)axes.flat[-1].legend(*axes.flat[0].get_legend_handles_labels(), loc='center', fontsize=12)axes.flat[-1].axis('off')plt.tight_layout()plt.savefig("national_residuals.png", dpi=150, bbox_inches='tight')plt.show()

---# Part 2: District-Level Population Prediction- 25 districts, 11 yearly data points each (2014-2024)- Uses **proportion-based approach**: predict (district/national) ratio- Split: Train (2014-2020), Val (2021-2022), Test (2023-2024)

In [ ]:
# ============================================================# CELL 15: Load District Data# ============================================================dist_df = pd.read_csv(f"{RAW_DIR}/sri_lanka_district_population_2014_2024_new.csv")dist_df = dist_df.groupby(["District", "Year"])["Total"].sum().reset_index()dist_df.rename(columns={"Total": "District_Population"}, inplace=True)# Merge national population for proportion calculationdist_df = dist_df.merge(nat_df[["Year", "National_Population"]], on="Year", how="left")dist_df["Proportion"] = dist_df["District_Population"] / dist_df["National_Population"]districts = sorted(dist_df["District"].unique())print(f"✅ District data loaded: {len(districts)} districts")print(f"   Years: {dist_df['Year'].min()}-{dist_df['Year'].max()}")print(f"   Districts: {districts}")

In [ ]:
# ============================================================# CELL 16: District Train/Val/Test Split & Model Training# ============================================================district_models = {}district_predictions = {}district_metrics = []for district in districts:    d = dist_df[dist_df["District"] == district].sort_values("Year").reset_index(drop=True)    # Split: Train 2014-2020, Val 2021-2022, Test 2023-2024    d_train = d[d["Year"] <= 2020]    d_val = d[(d["Year"] >= 2021) & (d["Year"] <= 2022)]    d_test = d[d["Year"] >= 2023]    if len(d_train) < 3 or len(d_val) < 1 or len(d_test) < 1:        continue    X_tr = d_train[["Year"]].values    y_tr = d_train["Proportion"].values    X_v = d_val[["Year"]].values    y_v = d_val["Proportion"].values    X_te = d_test[["Year"]].values    y_te = d_test["Proportion"].values    # Train Linear model on proportion    lr_d = LinearRegression()    lr_d.fit(X_tr, y_tr)    # Train Ridge model    ridge_d = Ridge(alpha=1.0)    ridge_d.fit(X_tr, y_tr)    # Pick best by validation    lr_val = lr_d.predict(X_v)    ridge_val = ridge_d.predict(X_v)    lr_val_mape = calc_mape(y_v, lr_val)    ridge_val_mape = calc_mape(y_v, ridge_val)    if lr_val_mape <= ridge_val_mape:        best_model, best_name = lr_d, "Linear"    else:        best_model, best_name = ridge_d, "Ridge"    preds = {        "train": best_model.predict(X_tr),        "val": best_model.predict(X_v),        "test": best_model.predict(X_te),    }    district_models[district] = {"model": best_model, "name": best_name}    district_predictions[district] = {        "years_train": d_train["Year"].values, "y_train": y_tr,        "years_val": d_val["Year"].values, "y_val": y_v,        "years_test": d_test["Year"].values, "y_test": y_te,        "preds": preds,    }    for split, yt, yp in [("Train", y_tr, preds["train"]), ("Val", y_v, preds["val"]), ("Test", y_te, preds["test"])]:        m = eval_metrics(yt, yp, y_tr, seasonal=1)        m.update({"District": district, "Model": best_name, "Split": split})        district_metrics.append(m)print(f"✅ Trained models for {len(district_models)} districts")

In [ ]:
# ============================================================# CELL 17: District Train/Val/Test Plots (Grid of all 25 districts)# ============================================================n_dist = len(district_predictions)ncols = 5nrows = (n_dist + ncols - 1) // ncolsfig, axes = plt.subplots(nrows, ncols, figsize=(28, nrows*4))fig.suptitle("District Population Proportions — Train / Val / Test", fontsize=18, fontweight='bold')for idx, district in enumerate(sorted(district_predictions.keys())):    ax = axes.flat[idx]    dp = district_predictions[district]    ax.plot(dp["years_train"], dp["y_train"]*100, 'bo-', markersize=5, label="Train")    ax.plot(dp["years_val"], dp["y_val"]*100, 'o-', color='orange', markersize=6, label="Val")    ax.plot(dp["years_test"], dp["y_test"]*100, 'ro-', markersize=6, label="Test")    ax.plot(dp["years_train"], dp["preds"]["train"]*100, 'b--', alpha=0.6)    ax.plot(dp["years_val"], dp["preds"]["val"]*100, '--', color='orange', linewidth=2)    ax.plot(dp["years_test"], dp["preds"]["test"]*100, 'r--', linewidth=2)    test_mape = calc_mape(dp["y_test"], dp["preds"]["test"])    ax.set_title(f"{district} (MAPE={test_mape:.2f}%)", fontsize=9, fontweight='bold')    ax.tick_params(labelsize=7)    ax.grid(True, alpha=0.3)# Hide unused axesfor idx in range(n_dist, len(axes.flat)):    axes.flat[idx].axis('off')# Add single legendaxes.flat[0].legend(fontsize=7)plt.tight_layout()plt.savefig("district_all_comparison.png", dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# ============================================================# CELL 18: District Metrics Table# ============================================================dist_metrics_df = pd.DataFrame(district_metrics)[["District", "Model", "Split", "RMSE", "MAE", "MAPE", "MASE", "R²"]]print("📊 District Population — Summary (Average Across Districts)")summary = dist_metrics_df.groupby("Split")[["RMSE", "MAE", "MAPE", "R²"]].mean()display(summary.style.format({"RMSE": "{:.6f}", "MAE": "{:.6f}", "MAPE": "{:.2f}%", "R²": "{:.4f}"}))print("\n📊 Per-District Test Metrics:")test_only = dist_metrics_df[dist_metrics_df["Split"] == "Test"].sort_values("MAPE")display(test_only.style.format({"RMSE": "{:.6f}", "MAE": "{:.6f}", "MAPE": "{:.2f}%", "MASE": "{:.3f}", "R²": "{:.4f}"}))dist_metrics_df.to_csv("district_metrics.csv", index=False)

---# Part 3: Prediction FunctionGiven a **date** and **district**, predict:1. National population for that date2. District population for that date

In [ ]:
# ============================================================# CELL 19: Prediction Function# ============================================================def predict_population(date_str, district_name=None):    """Predict national & district population for a given date."""    date = pd.to_datetime(date_str)    year = date.year    month = date.month    fractional_year = year + (month - 1) / 12.0    # Build feature vector (use latest known values for drivers if year > max)    row = nat_df[nat_df["Year"] <= year]    if len(row) == 0:        row = nat_df.head(1)    last_known = row.iloc[-1]    features = {}    for col in feature_cols:        if col == "Year":            features[col] = fractional_year        elif col in last_known.index:            features[col] = last_known[col]        else:            features[col] = 0    X = np.array([[features[c] for c in feature_cols]])    # Predict with all national models    results = {"date": date_str}    nat_preds = []    for name, model in models.items():        if name in ["Linear"]:            pred = model.predict(X)[0]        elif name in ["Ridge", "SVR"]:            pred = model.predict(scaler.transform(X))[0]        elif name in ["ARIMA"]:            steps = year - int(train_nat["Year"].max())            if steps > 0:                pred = arima_refit.predict(n_periods=steps)[-1]            else:                pred = y_train[-1]        elif name in ["ETS"]:            steps = year - int(train_nat["Year"].max())            if steps > 0:                pred = ets_refit.forecast(steps)[-1]            else:                pred = y_train[-1]        else:            continue        results[f"National_{name}"] = int(pred)        nat_preds.append(pred)    national_ensemble = np.mean(nat_preds)    results["National_Ensemble"] = int(national_ensemble)    # District prediction    if district_name and district_name in district_models:        dm = district_models[district_name]        prop = dm["model"].predict([[fractional_year]])[0]        dist_pop = prop * national_ensemble        results["District"] = district_name        results["District_Proportion"] = f"{prop:.4%}"        results["District_Population"] = int(dist_pop)    return results# Test predictionprint("=" * 60)print("🔮 PREDICTION EXAMPLES")print("=" * 60)# Example 1: National onlyr1 = predict_population("2025-06-01")print(f"\n📅 Date: 2025-06-01 (National Only)")for k, v in r1.items():    if k != "date":        print(f"   {k}: {v:,}" if isinstance(v, int) else f"   {k}: {v}")# Example 2: With districtr2 = predict_population("2025-06-01", "Colombo")print(f"\n📅 Date: 2025-06-01, District: Colombo")for k, v in r2.items():    if k != "date":        print(f"   {k}: {v:,}" if isinstance(v, int) else f"   {k}: {v}")# Example 3: Future dater3 = predict_population("2026-12-01", "Gampaha")print(f"\n📅 Date: 2026-12-01, District: Gampaha")for k, v in r3.items():    if k != "date":        print(f"   {k}: {v:,}" if isinstance(v, int) else f"   {k}: {v}")

In [ ]:
# ============================================================# CELL 20: Save All Models for Downstream Use# ============================================================import picklesave_dir = "population_models"os.makedirs(save_dir, exist_ok=True)# Save national modelsfor name, model in models.items():    with open(f"{save_dir}/national_{name.lower()}_model.pkl", "wb") as f:        pickle.dump(model, f)# Save scalerwith open(f"{save_dir}/feature_scaler.pkl", "wb") as f:    pickle.dump(scaler, f)# Save district modelswith open(f"{save_dir}/district_models.pkl", "wb") as f:    pickle.dump(district_models, f)# Save feature columnswith open(f"{save_dir}/feature_columns.pkl", "wb") as f:    pickle.dump(feature_cols, f)# Save national data (needed for prediction)nat_df.to_csv(f"{save_dir}/national_yearly_data.csv", index=False)# Save metricsmetrics_df.to_csv(f"{save_dir}/national_metrics.csv", index=False)dist_metrics_df.to_csv(f"{save_dir}/district_metrics.csv", index=False)print("✅ All models and data saved to:", save_dir)print("\nFiles saved:")for f in sorted(os.listdir(save_dir)):    size = os.path.getsize(f"{save_dir}/{f}")    print(f"  📁 {f} ({size:,} bytes)")print("\n💡 To use these models in your master pipeline:")print("   1. Download the 'population_models' folder")print("   2. Load with pickle.load()")print("   3. Call predict_population(date, district)")

In [ ]:
# ============================================================# CELL 21: Predict All Districts for a Given Date# ============================================================target_date = "2025-06-01"  # Change this to any date you wantprint(f"🗓️  Population Predictions for {target_date}")print("=" * 70)nat_result = predict_population(target_date)print(f"\n🇱🇰 National Population (Ensemble): {nat_result['National_Ensemble']:,}")print(f"\n{'District':<20} {'Proportion':>12} {'Population':>15}")print("-" * 50)all_results = []for district in sorted(districts):    r = predict_population(target_date, district)    if "District_Population" in r:        all_results.append(r)        print(f"{district:<20} {r['District_Proportion']:>12} {r['District_Population']:>15,}")print("-" * 50)total = sum(r["District_Population"] for r in all_results)print(f"{'TOTAL':<20} {'':>12} {total:>15,}")